## 4. Logistic regression model
Quantify each driver's *independent* contribution to churn — i.e. its effect holding all other variables constant.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/Churn_Analysis.csv').copy()
model_df = df.copy()

%load_ext autoreload
%autoreload 2

### 4.1 Prepare features
Drop non-predictive columns and one-hot encode categorical variables.

In [2]:
model_df = model_df.drop(columns=["customerID", "Churn", "Churn_binary", "Tenure_Tier"])

model_df = pd.get_dummies(model_df, drop_first=True)

print(f"Columns:\n{model_df.columns}")
print(f"\nRows: {model_df.shape[0]}, and columns: {model_df.shape[1]}\n")
print(model_df.head(4))

Columns:
Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='str')

Rows: 7043, and columns: 30

   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  gender_Male  \
0              0     

### 4.2 Train/test split
Hold out 20% to validate the model isn't just memorising training data.

In [3]:
X = model_df
y = df["Churn_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}, and Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f}")
print(f"Test churn rate: {y_test.mean():.3f}")

Train: (5634, 30), and Test: (1409, 30)
Train churn rate: 0.265
Test churn rate: 0.265


### 4.3 Fit the model

In [9]:
model = LogisticRegression(max_iter=3000, random_state=42)
model.fit(X_train, y_train)

train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)
print(f"Train accuracy: {train_accuracy:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")

Train accuracy: 0.806
Test accuracy: 0.807


### 4.4 Interpret coefficients
Rank features by impact; positive = increases churn risk, negative = decreases it.

In [10]:
coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_[0],
    "odds_ratio": np.exp(model.coef_[0]),
})
coefficients["abs_coef"] = coefficients["coefficient"].abs()
coefficients = coefficients.sort_values(by="abs_coef", ascending=False).drop(columns="abs_coef")

print(coefficients.to_string(index=False))

                              feature  coefficient  odds_ratio
                    Contract_Two year    -1.326554    0.265390
          InternetService_Fiber optic     1.316821    3.731540
                    Contract_One year    -0.683788    0.504702
                  StreamingMovies_Yes     0.431543    1.539632
                      StreamingTV_Yes     0.431065    1.538896
                    MultipleLines_Yes     0.390830    1.478207
       PaymentMethod_Electronic check     0.381809    1.464932
                 PaperlessBilling_Yes     0.375344    1.455492
                   OnlineSecurity_Yes    -0.322185    0.724564
                      TechSupport_Yes    -0.273044    0.761059
       MultipleLines_No phone service     0.271304    1.311674
                       Dependents_Yes    -0.221759    0.801108
   OnlineSecurity_No internet service    -0.193784    0.823836
                   InternetService_No    -0.193784    0.823836
 DeviceProtection_No internet service    -0.193784    0

In [11]:
coefficients.to_csv("../outputs/model_coefficients.csv", index=False)